# Lesson 9.2 · Hyperparameters and early stopping

XGBoost has many hyperparameters, but few make a difference. This lesson gives you the map and shows how to choose them efficiently with **early stopping**: stopping the training when the error on a validation set stops dropping.

**What you will learn**

- The groups of hyperparameters: step (`learning_rate`), tree complexity, sampling, regularization.
- How to use `eval_set` and `early_stopping_rounds`, and what never to do.
- Why the choice of the rounds is made on a validation set **different from the test**.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(4000, n_features=12, n_informative=5, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.4, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
print("training", X_tr.shape, "| validation", X_val.shape, "| test", X_te.shape)

## The map of the hyperparameters

| group | parameters (XGBoost) | what they do |
|---|---|---|
| step | `learning_rate` (eta), `n_estimators` | how much each tree weighs and how many are needed |
| complexity | `max_depth`, `min_child_weight`, `gamma` | depth and minimum "weight" (for the logistic: sum of the hessians) to split; `gamma` = minimum gain |
| sampling | `subsample`, `colsample_bytree` | fraction of rows and columns for each tree: they decorrelate and regularize (as in the forest) |
| regularization | `reg_lambda` (L2), `reg_alpha` (L1) | penalty on the leaf values |

A robust starting rule: `learning_rate=0.05`, `max_depth=4`, `subsample=0.8`, `colsample_bytree=0.8`, `min_child_weight=1`, and the rounds chosen with early stopping.

## Early stopping

You pass an `eval_set` (validation) and `early_stopping_rounds=k`: the training stops if the validation metric does not improve for `k` consecutive rounds, and the model uses the **best round**.

In [ ]:
model = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8,
                          eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)
model.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)
curves = model.evals_result()
print(f"rounds run: {len(curves['validation_1']['logloss'])} | best round: {model.best_iteration} | validation log-loss at the best: {model.best_score:.4f}")
print(f"training log-loss at the best round: {curves['validation_0']['logloss'][model.best_iteration]:.4f} | at the end: {curves['validation_0']['logloss'][-1]:.4f}")
print(f"AUC on the test: {roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]):.4f}")

The **training** log-loss keeps dropping for all the rounds, the **validation** one reaches a minimum and climbs back: that is the overfitting gap. Early stopping stops 30 rounds after the minimum and restores the best round.

**Golden rule:** the set used for early stopping is a **validation** set (part of the training data), not the test set: otherwise you chose the number of rounds by looking at the test and the generalization estimate is optimistic.

**Two details that raise no errors and ruin the model** (they are the bugs of exercises E1 and E2):

- With several sets in `eval_set`, the stopping is based on the **last** one. Here the training is first and the validation is last, which is the right order: putting the training last makes its log-loss always drop and the model never stops (E1).
- `best_iteration` does not truncate the booster: after `fit` it contains **all** the trained trees, including the 30 of "patience". `predict` and `predict_proba` of the wrapper use only the best ones; the native API (`Booster.predict`, `pred_contribs`) uses everything if you do not pass `iteration_range` (E2).

In [ ]:
booster = model.get_booster()
print(f"best_iteration = {model.best_iteration} -> good trees {model.best_iteration + 1}, trees in the booster {booster.num_boosted_rounds()}")
p_wrapper = model.predict_proba(X_te)[:, 1]
p_native = booster.predict(xgb.DMatrix(X_te))                                                            # all the trees
p_right = booster.predict(xgb.DMatrix(X_te), iteration_range=(0, model.best_iteration + 1))              # only the best ones
print(f"maximum gap from the wrapper: booster without iteration_range {np.abs(p_wrapper - p_native).max():.4f} | with iteration_range {np.abs(p_wrapper - p_right).max():.1e}")

> **Common mistake.** Changing the `learning_rate` without recalibrating the rounds. Halving eta requires about twice the rounds to get the same complexity: with a fixed `n_estimators` the model becomes undertrained. The curves of exercise D1 show how the round of the minimum changes as the configuration changes.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why is a small learning rate usually better?</summary>

Because each tree corrects little and the sum of many small steps is smoother and less prone to memorizing: the generalization improves (up to a limit) at the price of more rounds and more time. In practice you choose 0.02-0.1 and let early stopping decide the rounds.

</details>

<details><summary>What does `min_child_weight` do?</summary>

It prevents creating leaves with little 'weight' (the sum of the hessians: for regression it is the number of examples, for the logistic it is Σ p(1−p), so very confident examples count less). Higher values = more conservative trees, regularization against overfitting on few examples.

</details>